# Session 5: Advanced RAG & Evaluation

**Goal:** Enhance your RAG pipeline with hybrid search, reranking, and rigorous evaluation.

| What you'll learn | Why it matters |
|---|---|
| Hybrid search (BM25 + semantic) | Neither keyword nor semantic search alone is enough |
| Cross-encoder reranking | Precision layer for top candidates |
| Query rewriting | LLM reformulates queries for better retrieval |
| Confidence thresholds & citations | Production-quality answers with source attribution |
| Retrieval metrics (P@K, R@K, MRR) | Measure what you retrieve |
| LLM-as-Judge evaluation | Structured answer quality assessment |

**Corpus:** ~253,000 chunks from 60,000+ Croatian Wikipedia articles (pre-embedded, hosted).

**Time:** ~45 minutes

## Setup

In [1]:
!pip install chromadb-client openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.1/790.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have op

In [2]:
import json
import requests
from typing import List, Dict
from collections import defaultdict
from openai import OpenAI
import chromadb

# ── API endpoint ──
BASE_URL = "https://llmapi.levara.xyz"
API_KEY  = "alumni-llm-workshop-2026-ZAMJENI"  # ← zamijeni s ključem koji dobiješ na radionici

# OpenAI-compatible client for chat, embeddings, reranking
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key=API_KEY)

CHAT_MODEL   = "cyankiwi/gemma-4-26B-A4B-it-AWQ-8bit"
EMBED_MODEL  = "jinaai/jina-embeddings-v5-text-small-retrieval"
RERANK_MODEL = "jinaai/jina-reranker-v3"

# ChromaDB HTTP client
chroma = chromadb.HttpClient(
    host="llmapi.levara.xyz",
    port=443,
    ssl=True,
    headers={"User-Agent": "workshop"},
)
collection = chroma.get_collection("hrwiki")

print(f"Chat model:      {CHAT_MODEL}")
print(f"Embed model:     {EMBED_MODEL}")
print(f"Rerank model:    {RERANK_MODEL}")
print(f"ChromaDB chunks: {collection.count():,}")
print(f"\u2713 Ready")

Chat model:      cyankiwi/gemma-4-26B-A4B-it-AWQ-8bit
Embed model:     jinaai/jina-embeddings-v5-text-small-retrieval
Rerank model:    jinaai/jina-reranker-v3
ChromaDB chunks: 253,525
✓ Ready


In [3]:
import re as _re

def chat(messages, temperature=1.0, max_tokens=4096):
    """Send a chat completion request with Gemma 4 sampling params."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=temperature,
        top_p=0.95,
        max_tokens=max_tokens,
        extra_body={
            "top_k": 64,
            "chat_template_kwargs": {"enable_thinking": True},
        },
    )
    content = response.choices[0].message.content
    # Strip any leaked <think> tags from output
    content = _re.sub(r"<think>.*?</think>\s*", "", content, flags=_re.DOTALL)
    return content


def get_embedding(text: str) -> List[float]:
    """Embed a single text via the vLLM endpoint."""
    response = client.embeddings.create(input=[text], model=EMBED_MODEL)
    return response.data[0].embedding


print("\u2713 chat() and get_embedding() defined")

✓ chat() and get_embedding() defined


### Semantic search helper

Our ChromaDB collection was built with pre-computed embeddings (no server-side embedding function).
So to query it, we need to **embed the query ourselves** and pass the vector.

Let's see both steps explicitly first, then wrap them into a helper.

In [4]:
# Step 1: embed the query
query = "Bašćanska ploča"
query_embedding = get_embedding(query)
print(f"Query embedding: {len(query_embedding)} dimensions")

# Step 2: search ChromaDB with the vector
raw = collection.query(query_embeddings=[query_embedding], n_results=3)

for doc, meta, dist in zip(raw["documents"][0], raw["metadatas"][0], raw["distances"][0]):
    print(f"\n  [{1 - dist:.3f}] {meta['title']} \u2014 {meta['section']}")
    print(f"          {doc[:100]}...")

Query embedding: 1024 dimensions

  [0.732] Crkva sv. Lucije na Krku — Bašćanska ploča
          Bašćanska ploča
mini|Izvorna Bašćanska ploča u auli HAZU. Bašćanska ploča izvorno je bila lijevi plu...

  [0.658] Bašćanska ploča — U umjetnosti
          U umjetnosti Krešimir Fribec napisao je djelo „Baščanska ploča” kantatu za sola, zbor i komorni orke...

  [0.643] Ivan Standl — Bašćanska ploča i slike Hrvatske
          Bašćanska ploča i slike Hrvatske
1865. odlazi na svoje prvo terensko snimanje u Jurandvor na Krku, k...


In [5]:
def semantic_search(query: str, top_k: int = 10) -> List[Dict]:
    """Embed query, search ChromaDB, return ranked results."""
    embedding = get_embedding(query)
    raw = collection.query(query_embeddings=[embedding], n_results=top_k)
    results = []
    for doc, meta, dist in zip(
        raw["documents"][0], raw["metadatas"][0], raw["distances"][0]
    ):
        results.append({
            "text": doc,
            "title": meta["title"],
            "section": meta["section"],
            "score": 1 - dist,  # cosine similarity
        })
    return results


print("\u2713 semantic_search() defined")

✓ semantic_search() defined


## Part 1: Hybrid Search

Semantic search finds **meaning** — it understands synonyms, paraphrases, cross-language concepts.

But it struggles with **exact terms**: specific names, dates, identifiers.

BM25 (keyword search) is the opposite: great at exact matches, blind to meaning.

**Hybrid search** combines both.

### Where semantic search fails

In [6]:
# A query with a specific historical term + date
query = "Pacta Conventa 1102"

semantic_results = semantic_search(query, top_k=3)
print(f"Semantic search: '{query}'\n")
for i, r in enumerate(semantic_results):
    print(f"  {i+1}. [{r['score']:.3f}] {r['title']} \u2014 {r['section']}")
    print(f"     {r['text'][:120]}...\n")

Semantic search: 'Pacta Conventa 1102'

  1. [0.468] Pakt iz katakombi — Uvod
     Pakt iz katakombi za siromašnu Crkvu koja služi naziv je dokumenta koji potpisali 42 biskupa i kardinala, 16. studenoga ...

  2. [0.438] Hrvatska pravna povijest — Pacta conventa
     Pacta conventa
U sporazumu s hrvatskim velikašima Koloman se obvezao da će poštovati njihova tradicionalna prava, garant...

  3. [0.409] Striktna opservancija — Wilhelmsbadski konvent i raspad sustava: 1782.
     Wilhelmsbadski konvent i raspad sustava: 1782. Još prije pojave pustolova poput Alessandra Cagliostra i grofa Saint Germ...



In [7]:
def bm25_search(query: str, n_results: int = 10) -> List[Dict]:
    """Keyword search via the hosted BM25 endpoint."""
    response = requests.post(
        f"{BASE_URL}/v1/bm25",
        json={"query": query, "n_results": n_results},
        headers={"User-Agent": "workshop"},
    )
    response.raise_for_status()
    return response.json()["results"]


# Same query — BM25 nails the exact term
bm25_results = bm25_search(query, n_results=3)
print(f"BM25 search: '{query}'\n")
for i, r in enumerate(bm25_results):
    print(f"  {i+1}. [{r['score']:.1f}] {r['title']} \u2014 {r['section']}")
    print(f"     {r['text'][:120]}...\n")

BM25 search: 'Pacta Conventa 1102'

  1. [35.9] Hrvatska povijest — Personalna unija i Pacta conventa (1102.)
     Personalna unija i Pacta conventa (1102.) mini|100px|Pacta conventa mini|lijevo|160px|Oton Iveković: Smrt kralja Petra S...

  2. [32.6] Starohrvatska umjetnost — Povijesna pozadina
     ugo su zadržali svoje romansko obilježje i stanovništvo. Povremeno su padali pod vlast Mlečana ili Bizanta. Hrvati postu...

  3. [32.3] Hrvatska pravna povijest — Pacta conventa
     Pacta conventa
U sporazumu s hrvatskim velikašima Koloman se obvezao da će poštovati njihova tradicionalna prava, garant...



### Where BM25 fails

In [8]:
# A conceptual query — the corpus uses Croatian terms, not English
query = "Kako se Hrvatska branila od Osmanlija?"

print(f"BM25 search: '{query}'\n")
bm25_results = bm25_search(query, n_results=3)
for i, r in enumerate(bm25_results):
    print(f"  {i+1}. [{r['score']:.1f}] {r['title']} \u2014 {r['section']}")
    print(f"     {r['text'][:120]}...\n")

print("\n" + "=" * 80 + "\n")

print(f"Semantic search: '{query}'\n")
semantic_results = semantic_search(query, top_k=3)
for i, r in enumerate(semantic_results):
    print(f"  {i+1}. [{r['score']:.3f}] {r['title']} \u2014 {r['section']}")
    print(f"     {r['text'][:120]}...\n")

BM25 search: 'Kako se Hrvatska branila od Osmanlija?'

  1. [20.0] Hrvatske zemlje pod osmanskom vlašću — Osmanlijska opasnost i agresija
     Osmanlijska opasnost i agresija Godine 1433. kralj Žigmund je shvatio da je velika opasnost zaprijetila njegovim kraljev...

  2. [18.4] Vaterpolo na Univerzijadi 2019. — Uvod
     Muški vaterpolski turnir na 27. izdanju Univerzijade održao se od 2. do 14. srpnja 2019. godine u Napulju u Italiji. Srb...

  3. [17.9] Europsko prvenstvo u vaterpolu za žene – Barcelona 2018. — Uvod
     Europsko prvenstvo u vaterpolu za žene – Barcelona 2018. 16. je izdanje ovog natjecanja koje se održalo u Barceloni od 1...



Semantic search: 'Kako se Hrvatska branila od Osmanlija?'

  1. [0.720] Hrvati Bosne i Hercegovine — Osmansko Carstvo
     Osmansko Carstvo mini|lijevo|220 px|Stradanje i raseljavanje Hrvata katolika od strane Osmanlija
Tijekom osmanskog razdo...

  2. [0.699] Kornica — Osmansko Carstvo
     Osmansko Carstvo Tijekom osmanskog razdoblja i int

### Reciprocal Rank Fusion (RRF)

Combine both result lists into one. Documents that rank high in **both** lists get the highest fused score.

$\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}$

$k = 60$ (standard default) — balances top vs lower ranks.

In [9]:
def rrf_fusion(semantic_results: List[Dict], bm25_results: List[Dict], k: int = 60) -> List[Dict]:
    """Reciprocal Rank Fusion — merge two ranked lists into one."""
    scores = defaultdict(float)
    docs = {}  # text -> full result dict
    sources = defaultdict(set)

    for rank, r in enumerate(semantic_results):
        key = r["text"]
        scores[key] += 1 / (k + rank + 1)
        docs[key] = r
        sources[key].add("semantic")

    for rank, r in enumerate(bm25_results):
        key = r["text"]
        scores[key] += 1 / (k + rank + 1)
        if key not in docs:
            docs[key] = r
        sources[key].add("bm25")

    fused = []
    for text in sorted(scores, key=scores.__getitem__, reverse=True):
        result = dict(docs[text])
        result["rrf_score"] = scores[text]
        result["sources"] = sources[text]
        fused.append(result)

    return fused


print("\u2713 rrf_fusion() defined")

✓ rrf_fusion() defined


In [10]:
def hybrid_search(query: str, top_k: int = 10, semantic_k: int = 20, bm25_k: int = 20) -> List[Dict]:
    """Hybrid search: semantic + BM25 + RRF fusion."""
    sem = semantic_search(query, top_k=semantic_k)
    bm = bm25_search(query, n_results=bm25_k)
    fused = rrf_fusion(sem, bm)
    return fused[:top_k]


print("\u2713 hybrid_search() defined")

✓ hybrid_search() defined


### Side-by-side comparison

In [11]:
queries = [
    "Pacta Conventa 1102",
    "Kako se Hrvatska branila od Osmanlija?",
    "Jela\u010di\u0107 revolucija 1848",
    "Zrinsko-frankopanska urota 1671",
]

for query in queries:
    sem = semantic_search(query, top_k=3)
    bm = bm25_search(query, n_results=3)
    hyb = hybrid_search(query, top_k=3)

    print(f"\n{'=' * 80}")
    print(f"Query: {query}\n")

    for label, results in [("Semantic", sem), ("BM25", bm), ("Hybrid", hyb)]:
        titles = [f"{r['title']}" for r in results[:3]]
        print(f"  {label:10s}: {' | '.join(titles)}")
    print()


Query: Pacta Conventa 1102

  Semantic  : Pakt iz katakombi | Hrvatska pravna povijest | Striktna opservancija
  BM25      : Hrvatska povijest | Starohrvatska umjetnost | Hrvatska pravna povijest
  Hybrid    : Hrvatska pravna povijest | Hrvatska povijest | Pakt iz katakombi


Query: Kako se Hrvatska branila od Osmanlija?

  Semantic  : Hrvati Bosne i Hercegovine | Kornica | Osmansko Carstvo
  BM25      : Hrvatske zemlje pod osmanskom vlašću | Vaterpolo na Univerzijadi 2019. | Europsko prvenstvo u vaterpolu za žene – Barcelona 2018.
  Hybrid    : Hrvatsko Kraljevstvo (1102. – 1526.) | Hrvati Bosne i Hercegovine | Hrvatske zemlje pod osmanskom vlašću


Query: Jelačić revolucija 1848

  Semantic  : Hrvatska pod Habsburgovcima | Hrvatska povijest | Hrvatska povijest
  BM25      : Hrvatska pod Habsburgovcima | 1848. | Hrvatska povijest
  Hybrid    : Hrvatska pod Habsburgovcima | Hrvatska povijest | Hrvatska u Revoluciji 1848.


Query: Zrinsko-frankopanska urota 1671

  Semantic  : Zrinsko-

### 🔬 Your turn

Try your own queries. Some ideas:
- A specific person's name + date
- A broad conceptual question in Croatian
- A question mixing both (name + concept)

In [ ]:
# ── Try your own query ──
your_query = "Nikola Šubi\u0107 Zrinski Siget 1566"  # ← change this

print("Semantic:")
for r in semantic_search(your_query, top_k=3):
    print(f"  [{r['score']:.3f}] {r['title']} \u2014 {r['section']}")

print("\nBM25:")
for r in bm25_search(your_query, n_results=3):
    print(f"  [{r['score']:.1f}] {r['title']} \u2014 {r['section']}")

print("\nHybrid:")
for r in hybrid_search(your_query, top_k=3):
    src = "+".join(r['sources'])
    print(f"  [{r['rrf_score']:.4f}] ({src}) {r['title']} \u2014 {r['section']}")

Semantic:
  [0.520] Opsada Sigeta — Uvod u bitku
  [0.418] Zrinski — Uvod
  [0.395] Opsada Sigeta — Prvi sukobi prije Sigeta

BM25:
  [49.6] Nikola Šubić Zrinski — Uvod
  [39.8] Zrinski — Uvod
  [32.5] Ivan I. Drašković — O njemu

Hybrid:
  [0.0323] (bm25+semantic) Zrinski — Uvod
  [0.0305] (bm25+semantic) Nikola Šubić Zrinski — Životopis
  [0.0301] (bm25+semantic) Opsada Sigeta — Uvod u bitku


## Part 2: Reranking

Hybrid search gives us good **candidates**. But the ranking is approximate.

A **cross-encoder** (reranker) sees the query and each document **together** in one pass.
It deeply understands whether a document actually answers the question — not just whether it's similar.

| Method | How it works | Speed | Precision |
|---|---|---|---|
| Bi-encoder (search) | Encode separately, compare vectors | Fast | Good |
| Cross-encoder (rerank) | Encode query+doc together | Slow | Excellent |

That's why we use **two stages**: fast search for candidates, then precise reranking.

In [12]:
def rerank(query: str, documents: List[str], top_n: int = 5) -> List[Dict]:
    """Rerank documents using the cross-encoder. Returns sorted results."""
    response = requests.post(
        f"{BASE_URL}/v1/rerank",
        json={
            "model": RERANK_MODEL,
            "query": query,
            "documents": documents,
            "top_n": top_n,
        },
        headers={"User-Agent": "workshop"},
    )
    response.raise_for_status()
    return response.json()["results"]


print("\u2713 rerank() defined")

✓ rerank() defined


In [13]:
# Retrieve 20 candidates, then rerank to top 5
query = "Tko je bio Petar Kru\u017ei\u0107 i kako je branio Klis?"

candidates = hybrid_search(query, top_k=20)
doc_texts = [r["text"] for r in candidates]

reranked = rerank(query, doc_texts, top_n=5)

print(f"Query: {query}\n")
print(f"Before reranking (top 5 of {len(candidates)} candidates):")
for i, r in enumerate(candidates[:5]):
    print(f"  {i+1}. {r['title']} \u2014 {r['section']}")

print(f"\nAfter reranking (top 5):")
for rr in reranked:
    orig_rank = rr["index"] + 1
    title = candidates[rr["index"]]["title"]
    section = candidates[rr["index"]]["section"]
    move = orig_rank - (reranked.index(rr) + 1)
    arrow = f"\u2191{move}" if move > 0 else (f"\u2193{-move}" if move < 0 else "=")
    print(f"  [{rr['relevance_score']:.3f}] {title} \u2014 {section}  (was #{orig_rank} {arrow})")

Query: Tko je bio Petar Kružić i kako je branio Klis?

Before reranking (top 5 of 20 candidates):
  1. Petar Kružić — Vojno djelovanje
  2. Petar Kružić — Podrijetlo
  3. Petar Kružić — Uvod
  4. Petar Kružić — Vojno djelovanje
  5. Grgur Orlovčić — Međusobna adopcija s Petrom Kružićem

After reranking (top 5):
  [0.427] Petar Kružić — Uvod  (was #3 ↑2)
  [0.336] Petar Kružić — Vojno djelovanje  (was #1 ↓1)
  [0.235] Petar Kružić — Vojno djelovanje  (was #4 ↑1)
  [0.205] Petar Kružić — Vojno djelovanje  (was #13 ↑9)
  [0.118] Antemurale Christianitatis — Uvod  (was #15 ↑10)


In [14]:
def retrieve_and_rerank(query: str, candidates: int = 20, top_n: int = 3) -> List[Dict]:
    """Two-stage retrieval: hybrid search → cross-encoder reranking."""
    results = hybrid_search(query, top_k=candidates)
    doc_texts = [r["text"] for r in results]
    reranked = rerank(query, doc_texts, top_n=top_n)

    final = []
    for rr in reranked:
        idx = rr["index"]
        entry = dict(results[idx])
        entry["rerank_score"] = rr["relevance_score"]
        final.append(entry)
    return final


print("\u2713 retrieve_and_rerank() defined")

✓ retrieve_and_rerank() defined


### 🔬 Your turn

Experiment with different candidate pool sizes and top_n values.

In [15]:
query = "Bitka na Krbavskom polju"  # ← change this

for candidates, top_n in [(10, 3), (20, 3), (20, 5), (50, 3)]:
    results = retrieve_and_rerank(query, candidates=candidates, top_n=top_n)
    titles = [f"{r['title']} ({r['rerank_score']:.2f})" for r in results]
    print(f"  candidates={candidates}, top_n={top_n}: {' | '.join(titles)}")

  candidates=10, top_n=3: Krbavska bitka (0.40) | Emerik Derenčin (0.26) | Krbavska bitka (0.13)
  candidates=20, top_n=3: Krbavska bitka (0.41) | Emerik Derenčin (0.27) | Krbavska bitka (0.13)
  candidates=20, top_n=5: Krbavska bitka (0.41) | Emerik Derenčin (0.27) | Krbavska bitka (0.13) | Krbavsko polje (0.11) | Krbavska bitka (0.09)
  candidates=50, top_n=3: Krbavska bitka (0.44) | Emerik Derenčin (0.27) | Krbavska bitka (0.13)


## Part 3: Production Patterns

Now we chain everything into a production-quality RAG pipeline:

```
Question → Rewrite → Hybrid Search → Rerank → Confidence Check → Generate with Citations
```

### Query rewriting

The user's query may be vague, abbreviated, or colloquial.
An LLM can reformulate it into a retrieval-optimized form.

In [16]:
def rewrite_query(query: str) -> str:
    """Use the LLM to reformulate a query for better retrieval."""
    response = chat([
        {"role": "system", "content": (
            "You are a search query rewriter for a Croatian Wikipedia knowledge base. "
            "Rewrite the user's query to improve retrieval. "
            "Expand abbreviations, add relevant context terms, keep it in Croatian. "
            "When expanding the abbreviations, keep both the original abbreviation and the expaned term in the final query."
            "Respond ONLY with the rewritten query, nothing else."
        )},
        {"role": "user", "content": query},
    ])
    return response.strip()


# Demo
original = "NDH"
rewritten = rewrite_query(original)
print(f"Original:  {original}")
print(f"Rewritten: {rewritten}")

print(f"\nRetrieval with original:")
for r in retrieve_and_rerank(original, top_n=3):
    print(f"  [{r['rerank_score']:.3f}] {r['title']}")

print(f"\nRetrieval with rewritten:")
for r in retrieve_and_rerank(rewritten, top_n=3):
    print(f"  [{r['rerank_score']:.3f}] {r['title']}")

Original:  NDH
Rewritten: Nezavisna Država Hrvatska (NDH)

Retrieval with original:
  [0.225] Povijest Nezavisne Države Hrvatske (Matković)
  [0.148] Džafer Kulenović
  [0.140] Nezavisna Država Hrvatska

Retrieval with rewritten:
  [0.401] Nezavisna Država Hrvatska
  [0.230] Hrvatska povijest
  [0.172] Povijest Bosne i Hercegovine


### Confidence thresholds

The reranker score tells us how confident we are in the retrieved context.
Instead of always answering (and risking hallucination), we can **decide** based on score.

| Score | Action |
|---|---|
| > 0.5 | ✅ High confidence — answer normally |
| 0.2 – 0.5 | ⚠️ Medium — answer with caveat |
| < 0.2 | ❌ Low — decline to answer |

### Citation-aware generation

We format the context with numbered source labels `[1]`, `[2]`, and instruct the LLM to cite them.

In [17]:
SYSTEM_PROMPT = """You are a helpful assistant answering questions about Croatian history and culture.

RULES:
1. Answer ONLY based on the provided context.
2. Cite sources using [1], [2], etc. corresponding to the numbered context passages.
3. If the context doesn't contain enough information, say: "Nemam dovoljno informacija u zadanom kontekstu."
4. Do NOT use knowledge from your training data — only the provided context.
5. Answer in the same language as the question."""


def enhanced_rag_query(
    question: str,
    use_rewrite: bool = True,
    candidates: int = 20,
    top_n: int = 5,
    high_threshold: float = 0.5,
    low_threshold: float = 0.2,
) -> Dict:
    """Full enhanced RAG pipeline."""
    # 1. Query rewriting
    rewritten = rewrite_query(question) if use_rewrite else question

    # 2. Hybrid search + reranking
    sources = retrieve_and_rerank(rewritten, candidates=candidates, top_n=top_n)

    # 3. Confidence check
    top_score = sources[0]["rerank_score"] if sources else 0
    if top_score >= high_threshold:
        confidence = "high"
    elif top_score >= low_threshold:
        confidence = "medium"
    else:
        confidence = "low"

    if confidence == "low":
        return {
            "answer": "Nemam dovoljno informacija u zadanom kontekstu za pouzdani odgovor.",
            "sources": sources,
            "confidence": confidence,
            "rewritten_query": rewritten,
        }

    # 4. Build prompt with numbered citations
    context = "\n\n".join(
        f"[{i+1}] Izvor: {s['title']} \u2014 {s['section']}\n{s['text']}"
        for i, s in enumerate(sources)
    )

    system = SYSTEM_PROMPT
    if confidence == "medium":
        system += "\n\nNote: confidence in the retrieved context is moderate. Indicate uncertainty where appropriate."

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Kontekst:\n\n{context}\n\nPitanje: {question}"},
    ]

    answer = chat(messages)

    return {
        "answer": answer,
        "sources": sources,
        "confidence": confidence,
        "rewritten_query": rewritten,
    }


print("\u2713 enhanced_rag_query() defined")

✓ enhanced_rag_query() defined


In [18]:
def show_result(question: str, result: Dict):
    """Pretty-print an enhanced RAG result."""
    print(f"Question:  {question}")
    if result["rewritten_query"] != question:
        print(f"Rewritten: {result['rewritten_query']}")
    print(f"Confidence: {result['confidence']}")
    print("=" * 80)
    print(f"\n{result['answer']}\n")
    print("-" * 80)
    print("Sources:")
    for i, s in enumerate(result["sources"]):
        print(f"  [{i+1}] [{s['rerank_score']:.3f}] {s['title']} \u2014 {s['section']}")


# Test the full pipeline
result = enhanced_rag_query("Tko je bio Petar Kru\u017ei\u0107 i kako je branio Klis?")
show_result("Tko je bio Petar Kru\u017ei\u0107 i kako je branio Klis?", result)

Question:  Tko je bio Petar Kružić i kako je branio Klis?
Rewritten: Petar Kružić, obrana tvrđave Klis, povijest Dalmacije, ratovi protiv Osmanlija, hrvatski plemić i vojni zapovjednik.
Confidence: medium

Petar Kružić bio je hrvatski vojskovođ, knez kliški te kapetan Klisa i Senja [3]. Svoju vojnu karijeru započeo je 1513. godine u Klisu obavljajući dužnost podkaštelana, a hrvatski ban ga je postavio za kapetana Klisa (1518. ili 1519. godine) [4]. Krajem 1521. godine imenovan je kapetanom Senja [4].

Klis je branio na sljedeće načine:

*   **Dugotrajni otpor:** Oduzio se turskoj sili tijekom više od dva desetljeća neprestanih opsada i krvavih bitaka [1].
*   **Odbijanje opsade iz 1524. godine:** Kada je turski vojvoda Mustafa opkolio Klis s 3000 vojnika, Kružić je u Senju okupio vojsku od 1500 pješaka i 60 konjanika (uz pomoć Toma Nigera), te je s 40 brodova stigao do Solina odakle je krenuo prema Klisu i razbio opsadu [4].
*   **Osloboditelj Klisa iz 1532. godine:** Nakon što je 4. l

In [19]:
# More test queries
for q in [
    "Što je bila Zrinsko-frankopanska urota?",
    "Kada je Dubrova\u010dka Republika ukinuta?",
]:
    result = enhanced_rag_query(q)
    show_result(q, result)
    print("\n")

Question:  Što je bila Zrinsko-frankopanska urota?
Rewritten: Zrinsko-frankopanska urota, povijest Hrvatske, 17. stoljeće, plemićka konspiracija, Nikola VII Zrinski, Fran Krsto Frankopan, Habsburgova monarhija.
Confidence: medium

Zrinsko-frankopanska urota bio je pokret hrvatskog i ugarskog plemstva protiv apsolutističke politike Habsburgovaca koji je trajao u razdoblju od 1664. do 1671. godine [2, 5].

Glavni razlozi i ciljevi urote bili su:
* **Odupiranje centralizmu i apsolutizmu:** Urotničari su željeli umanjiti sve veći centralizam i apsolutizam hrvatsko-ugarskog kralja Leopolda I. Habsburgovca [1]. Pokušali su se oduprijeti nametanju centralizma i apsolutizma kakvim se već upravljalo u Austrijskim nasljednim zemljama i Češkoj [2, 5].
* **Politički razlozi:** Izraženo je nezadovoljstvo politikom centralizacije i germanizacije [3].
* **Vašvarski mir:** Povod za nezadovoljstvo bio je sramotni Vašvarski mir sklopljen 10. kolovoza 1664. godine, kojim su Turci, premda poraženi u Monoš

In [20]:
# Out-of-domain grounding test — Kubernetes has zero mentions in the corpus
result = enhanced_rag_query("Objasni kako Kubernetes upravlja kontejnerima.")
show_result("Objasni kako Kubernetes upravlja kontejnerima.", result)
print("\n\u2192 The system should decline \u2014 this topic isn't in the corpus.")

Question:  Objasni kako Kubernetes upravlja kontejnerima.
Rewritten: Kubernetes orkestracija kontejnera, upravljanje kontejnerima, Pods, čvorovi (nodes), klasteri, Kubernetes arhitektura
Confidence: low

Nemam dovoljno informacija u zadanom kontekstu za pouzdani odgovor.

--------------------------------------------------------------------------------
Sources:
  [1] [0.043] Računarstvo — Arhitektura računala
  [2] [0.030] Mreža — Telekomunikacijske mreže
  [3] [-0.029] Lučka manipulacija — Pražnjenje kontejnera
  [4] [-0.092] Procesor (računarstvo) — Arhitektura
  [5] [-0.096] Vrtlarstvo — Kontejnersko vrtlarstvo

→ The system should decline — this topic isn't in the corpus.


## Part 4: Evaluation

How do we know if our RAG pipeline is actually good?

Two levels:

| Level | Question | Method |
|---|---|---|
| **Retrieval** | Did we find the right documents? | Precision, Recall, MRR |
| **Answer** | Is the answer correct? | LLM-as-Judge |

Remember: **cosine similarity measures retrieval, not correctness!**

### Test set

A mix of query types: exact terms, conceptual, dates, proper nouns.

In [21]:
test_set = [
    {
        "question": "Tko je bio Petar Kružić?",
        "expected_answer": "Hrvatski kapetan i knez koji je branio utvrdu Klis od Osmanlija. Poginuo je 1537. u pokušaju probijanja osmanske opsade.",
        "relevant_titles": ["Petar Kružić", "Klis"],
    },
    {
        "question": "Pacta Conventa 1102",
        "expected_answer": "Sporazum iz 1102. godine kojim je Hrvatska ušla u personalnu uniju s Ugarskom, zadržavši vlastitu upravu i sabor.",
        "relevant_titles": ["Pacta conventa", "Koloman"],
    },
    {
        "question": "Što je bila Zrinsko-frankopanska urota?",
        "expected_answer": "Urota hrvatskog i ugarskog plemstva protiv Habsburške vlasti 1664.-1671. Predvodili su je Petar Zrinski i Fran Krsto Frankopan, koji su pogubljeni 1671.",
        "relevant_titles": ["Zrinsko-frankopanska urota", "Petar IV. Zrinski"],
    },
    {
        "question": "Kako se Hrvatska branila od Osmanlija?",
        "expected_answer": "Hrvatska je bila predzidje kršćanstva (Antemurale Christianitatis). Ključne obrane uključuju Klis, Siget, i Vojnu krajinu.",
        "relevant_titles": ["Petar Kružić", "Nikola Šubić Zrinski", "Vojna krajina", "Klis", "Opsada Sigeta"],
    },
    {
        "question": "Jelačić revolucija 1848",
        "expected_answer": "Ban Josip Jelačić ukinuo je kmetstvo 1848. i poveo vojsku protiv Mađarske revolucije u obranu Habsburške monarhije.",
        "relevant_titles": ["Josip Jelačić", "Revolucija 1848. \u2013 1849."],
    },
    {
        "question": "Koji su najvažniji glagoljski spomenici?",
        "expected_answer": "Najvažniji glagoljski spomenici su Bašćanska ploča, Vinodolski zakonik, i Istarski razvod.",
        "relevant_titles": ["Bašćanska ploča", "Vinodolski zakonik", "Istarski razvod", "Glagoljica"],
    },
    {
        "question": "Kada je Dubrovačka Republika ukinuta?",
        "expected_answer": "Dubrovačka Republika ukinuta je 1808. godine za vrijeme Napoleonovih osvajanja.",
        "relevant_titles": ["Dubrovačka Republika"],
    },
    {
        "question": "Uloga bana Josipa Jelačića u ukidanju kmetstva",
        "expected_answer": "Jelačić je 25. travnja 1848. proglasio ukidanje kmetstva u Hrvatskoj, čime su seljaci oslobođeni feudalnih obveza.",
        "relevant_titles": ["Josip Jelačić"],
    },
    {
        "question": "Bitka na Krbavskom polju 1493",
        "expected_answer": "Težak poraz hrvatske vojske od Osmanlija 1493. kod Udbine. Poginuo je velik dio hrvatskog plemstva.",
        "relevant_titles": ["Krbavska bitka"],
    },
    {
        "question": "Kako je Hrvatska ušla u personalnu uniju s Ugarskom?",
        "expected_answer": "Nakon izumrća dinastije Trpimirovića, Koloman je 1102. krunjen za hrvatskog kralja, čime je započela personalna unija.",
        "relevant_titles": ["Pacta conventa", "Koloman", "Trpimirovići"],
    },
]

print(f"\u2713 Test set: {len(test_set)} questions")

✓ Test set: 10 questions


### Retrieval metrics

**Precision@K** — of K retrieved, how many are relevant?

**Recall@K** — of all relevant docs, how many did we find?

**MRR** (Mean Reciprocal Rank) — where does the first relevant result appear?

In [22]:
def is_relevant(result: Dict, relevant_titles: List[str]) -> bool:
    """Check if a retrieved chunk matches any relevant title."""
    title = result.get("title", "").lower()
    return any(rt.lower() in title or title in rt.lower() for rt in relevant_titles)


def precision_at_k(results: List[Dict], relevant_titles: List[str], k: int) -> float:
    top_k = results[:k]
    relevant = sum(1 for r in top_k if is_relevant(r, relevant_titles))
    return relevant / k


def recall_at_k(results: List[Dict], relevant_titles: List[str], k: int) -> float:
    top_k = results[:k]
    found = set()
    for r in top_k:
        for rt in relevant_titles:
            if rt.lower() in r.get("title", "").lower() or r.get("title", "").lower() in rt.lower():
                found.add(rt)
    return len(found) / len(relevant_titles) if relevant_titles else 0


def mrr(results: List[Dict], relevant_titles: List[str]) -> float:
    for i, r in enumerate(results):
        if is_relevant(r, relevant_titles):
            return 1 / (i + 1)
    return 0


print("\u2713 Retrieval metrics defined")

✓ Retrieval metrics defined


### Configuration comparison

Run all test questions through 4 configurations and compare retrieval quality.

In [23]:
K = 3  # evaluate at top-3

configs = {
    "Semantic only": lambda q: semantic_search(q, top_k=K),
    "BM25 only": lambda q: bm25_search(q, n_results=K),
    "Hybrid (RRF)": lambda q: hybrid_search(q, top_k=K),
    "Hybrid + Rerank": lambda q: retrieve_and_rerank(q, candidates=20, top_n=K),
}

print(f"Evaluating {len(configs)} configs on {len(test_set)} questions at K={K}...\n")

results_table = {}
for config_name, search_fn in configs.items():
    p_scores, r_scores, mrr_scores = [], [], []

    for test in test_set:
        results = search_fn(test["question"])
        p_scores.append(precision_at_k(results, test["relevant_titles"], K))
        r_scores.append(recall_at_k(results, test["relevant_titles"], K))
        mrr_scores.append(mrr(results, test["relevant_titles"]))

    results_table[config_name] = {
        "P@3": sum(p_scores) / len(p_scores),
        "R@3": sum(r_scores) / len(r_scores),
        "MRR": sum(mrr_scores) / len(mrr_scores),
    }

# Print comparison table
print(f"{'Config':<20} {'P@3':>6} {'R@3':>6} {'MRR':>6}")
print("-" * 42)
for name, metrics in results_table.items():
    print(f"{name:<20} {metrics['P@3']:>6.3f} {metrics['R@3']:>6.3f} {metrics['MRR']:>6.3f}")

Evaluating 4 configs on 10 questions at K=3...

Config                  P@3    R@3    MRR
------------------------------------------
Semantic only         0.400  0.525  0.600
BM25 only             0.333  0.483  0.550
Hybrid (RRF)          0.367  0.475  0.550
Hybrid + Rerank       0.533  0.708  0.833


### LLM-as-Judge

Retrieval metrics measure **what you find**, not **what you answer**.

For answer quality, we use an LLM as an evaluator. It scores each answer on:
- Factual accuracy
- Completeness
- Relevance
- Grounding (does it stick to context?)

And returns a verdict: CORRECT / PARTIALLY_CORRECT / INCORRECT

In [24]:
JUDGE_PROMPT = """You are an expert evaluator for a Croatian history QA system.

Given a question, the retrieved context, a reference answer, and the system's answer,
evaluate the system's answer.

Score each criterion 1-10:
- factual_accuracy: Are the stated facts correct based on the context?
- completeness: Does it cover the key points from the reference answer?
- relevance: Does it actually answer the question asked?
- grounding: Does it stick to the provided context (vs. hallucinating)?

Verdict: CORRECT / PARTIALLY_CORRECT / INCORRECT

Respond ONLY with valid JSON:
{"factual_accuracy": N, "completeness": N, "relevance": N, "grounding": N, "verdict": "...", "reasoning": "..."}"""


def judge_answer(question: str, context: str, expected: str, answer: str) -> Dict:
    """Use the LLM to evaluate a RAG answer."""
    user_msg = (
        f"Question: {question}\n\n"
        f"Retrieved context:\n{context}\n\n"
        f"Reference answer: {expected}\n\n"
        f"System answer: {answer}"
    )
    response = chat(
        [{"role": "system", "content": JUDGE_PROMPT}, {"role": "user", "content": user_msg}],
        temperature=0,
    )
    # Parse JSON from response (handle markdown code blocks)
    text = response.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(text)


print("\u2713 judge_answer() defined")

✓ judge_answer() defined


In [25]:
# Run the judge on a few test questions using the full enhanced pipeline
judge_questions = test_set[:4]  # first 4 questions

for test in judge_questions:
    result = enhanced_rag_query(test["question"])

    context = "\n---\n".join(
        f"{s['title']} \u2014 {s['section']}: {s['text'][:200]}..."
        for s in result["sources"]
    )

    judgment = judge_answer(
        test["question"], context, test["expected_answer"], result["answer"]
    )

    print(f"\nQ: {test['question']}")
    print(f"  Verdict: {judgment['verdict']}")
    print(f"  Accuracy: {judgment['factual_accuracy']}/10  "
          f"Completeness: {judgment['completeness']}/10  "
          f"Relevance: {judgment['relevance']}/10  "
          f"Grounding: {judgment['grounding']}/10")
    print(f"  Reasoning: {judgment.get('reasoning', '')}")


Q: Tko je bio Petar Kružić?
  Verdict: PARTIALLY_CORRECT
  Accuracy: 10/10  Completeness: 4/10  Relevance: 10/10  Grounding: 1/10
  Reasoning: The system answer is factually correct according to general historical knowledge, but it fails significantly on grounding because the provided context does not contain any information about Petar Kružić (it mentions Petar Berislavić instead). Additionally, the answer is incomplete compared to the reference answer, as it omits key details such as his defense of the Klis fortress and the circumstances of his death.

Q: Pacta Conventa 1102
  Verdict: PARTIALLY_CORRECT
  Accuracy: 10/10  Completeness: 10/10  Relevance: 10/10  Grounding: 4/10
  Reasoning: The system's answer is factually accurate, highly relevant, and very complete, covering all points in the reference answer and providing much deeper historical context. However, it fails significantly on the grounding criterion. A large portion of the detailed information (such as the coronation in

### 🔬 Your turn

Add your own test questions. Define the question, an expected answer, and which article titles are relevant.

In [26]:
# ── Add your own test questions ──
my_tests = [
    {
        "question": "Što se dogodilo u Vukovaru 1991.?",  # ← change these
        "expected_answer": "Bitka za Vukovar trajala je 87 dana. Grad je te\u0161ko razoren u srpskoj agresiji.",
        "relevant_titles": ["Bitka za Vukovar", "Vukovar"],
    },
    # Add more...
]

# Run evaluation
for test in my_tests:
    result = enhanced_rag_query(test["question"])

    # Retrieval metrics
    p = precision_at_k(result["sources"], test["relevant_titles"], 3)
    r = recall_at_k(result["sources"], test["relevant_titles"], 3)
    m = mrr(result["sources"], test["relevant_titles"])

    print(f"\nQ: {test['question']}")
    print(f"  Retrieval: P@3={p:.2f}  R@3={r:.2f}  MRR={m:.2f}")
    print(f"  Confidence: {result['confidence']}")
    print(f"  Answer: {result['answer'][:200]}...")

    # Judge
    context = "\n---\n".join(s['text'][:200] for s in result["sources"])
    judgment = judge_answer(test["question"], context, test["expected_answer"], result["answer"])
    print(f"  Judge: {judgment['verdict']} (accuracy={judgment['factual_accuracy']}, completeness={judgment['completeness']})")


Q: Što se dogodilo u Vukovaru 1991.?
  Retrieval: P@3=1.00  R@3=1.00  MRR=1.00
  Confidence: medium
  Answer: Tijekom 1991. godine u Vukovaru su se dogodili sljedeći događaji:

* **Opsada i bitka:** Od kolovoza do studenoga 1991. godine trajala je 87-dnevna opsada grada koju su sproveli Jugoslavenska narodna ...
  Judge: PARTIALLY_CORRECT (accuracy=10, completeness=10)


## Key Takeaways

| Concept | What you learned |
|---|---|
| **Hybrid search** | BM25 + semantic via RRF — neither alone is sufficient |
| **Reranking** | Cross-encoder precision on top candidates |
| **Query rewriting** | LLM reformulates queries for better retrieval |
| **Confidence thresholds** | Score-based decision: answer / caveat / decline |
| **Citations** | Numbered source references build trust and traceability |
| **Retrieval metrics** | P@K, R@K, MRR — measure what you retrieve |
| **LLM-as-Judge** | Structured evaluation for answer quality |
| **Configuration comparison** | Don't guess — measure which setup works for your data |